In [1]:
# Imports
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import timm
from torch.optim import AdamW
from tqdm import tqdm
from pathlib import Path

/home/cqilab/anaconda3/envs/llmfinetune/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Hyperparams
BATCH_SIZE = 8
NUM_EPOCHS = 150
LEARNING_RATE = 0.0005
NUM_CLASSES = 83

device = torch.device(f"cuda:1" if torch.cuda.is_available() else "cpu")

# Dataset path
csv_path = "./Dataset/vit_dataset.csv"

# ---------- label maps ----------
TYPE2IDX = {
    "Cat state":      0,
    "Coherent state": 1,
    "Thermal state":  2,
    "Fock state":     3,
    "Random state":   4,
    "Number state":   5,
}
#  6 + 30 + 16 + 14 + 11 + 6

# qubit values assumed to be 1‒30  ➜ map «value → class‑idx»
QUBIT2IDX = {q: (q - 1) for q in range(0, 31)}      # 30 classes (0‑29)

# alpha values assumed to be 0‒15  ➜ map «value → class‑idx»
ALPHA2IDX = {a: a for a in range(0, 16)}      # 16 classes (0‑15)

# photons values assumed to be 3‒15  ➜ map «value → class‑idx»
PHOTONS2IDX = {p: (p - 2) for p in range(3, 16)}      # 14 classes (0‑13)
PHOTONS2IDX[0] = 0

DENSITY2IDX = {(float(d/10)): d for d in range(0,11)}

LINSPACE2IDX = {l: (l-5) for l in range(5,11)}

# Dataset class multiclass classification
class WignerDataset(Dataset):
    def __init__(self, csv_file=None, dataframe=None, image_dir=None, transform=None):
        """
        Args:
            csv_file (str): Path to the CSV file
            image_dir (str): Root dir to prepend to image path if not included in CSV
            transform (callable, optional): Transform to apply on images (e.g., Resize, ToTensor)
        """
        if dataframe is not None:
            self.data = dataframe.reset_index(drop=True)
        elif csv_file is not None:
            self.data = pd.read_csv(csv_file)
        else:
            raise ValueError("Either csv_file or dataframe must be provided.")
        
        self.image_dir = image_dir
        self.transform = transform


    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Image loading
        img_path = row['image']
        if self.image_dir and not os.path.isabs(img_path):
            img_path = os.path.join(self.image_dir, img_path)
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Labels (modify depending on task)
        label = {
            'type': TYPE2IDX[row['type']],  # convert class label to int
            'number_of_qubit': QUBIT2IDX[row['number_of_qubit']],
            'alpha': ALPHA2IDX[row['alpha']],
            'number_of_photons': PHOTONS2IDX[row['number_of_photons']],
            'density': DENSITY2IDX[row['density']],
            'linspace': LINSPACE2IDX[row['linspace']],
        }

        return image, label

In [3]:
# Transformations
transforms = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


# Train test splitfrom sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(pd.read_csv(csv_path), test_size=0.2, random_state=42)
# train_df, test_df = train_test_split(pd.read_csv(csv_path), test_size=0.9, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)

train_data = WignerDataset(dataframe=train_df, image_dir="./Dataset/", transform=transforms)
val_data = WignerDataset(dataframe=val_df, image_dir="./Dataset/", transform=transforms)
test_data = WignerDataset(dataframe=test_df, image_dir="./Dataset/", transform=transforms)


# dataset = WignerDataset(csv_path, transform=transforms)

# Train dataLoader and test dataLoader
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

In [4]:
# System model using ViT-large-16-224
# Use timm
# Last layer should be 70
# Task is multiclass classification
# Use partition of nodes to determine each class, 0-6 (what highest prob), 7-36 (what highest prob), 37-47 (what highest prob), 
# 49-59 (what highest prob), 59-69 (what highest prob)
from transformers import CLIPModel, CLIPProcessor

class CLIPMultiClassPartitioned(nn.Module):
    def __init__(self, num_classes=83):
        super(CLIPMultiClassPartitioned, self).__init__()
        # Load pretrained CLIP vision model
        self.clip_model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14-336")
        self.vision_encoder = self.clip_model.vision_model

        # Projection layer to map CLIP output to desired class count
        self.classifier = nn.Sequential(
            nn.Linear(self.vision_encoder.config.hidden_size, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes)
        )

    def forward(self, pixel_values):
        vision_outputs = self.vision_encoder(pixel_values)
        pooled_output = vision_outputs.pooler_output 
        logits = self.classifier(pooled_output)
        return logits

    def partition_predictions(self, logits):
        """
        Partition logic:
        - [0-5]       -> Type (6)
        - [6-35]      -> Number of qubits (30)
        - [36-51]     -> Alpha (16)
        - [52-65]     -> Number of photons (14)
        - [66-76]     -> Density (11)
        - [77-82]     -> Linear Space (6)
        Returns dict with max-predicted class index for each partition
        """
        preds = {}
        preds['type'] = logits[:, 0:6]
        preds['number_of_qubit'] = logits[:, 6:36]
        preds['alpha'] = logits[:, 36:52]
        preds['number_of_photons'] = logits[:, 52:66]
        preds['density'] = logits[:, 66:77]
        preds['linspace'] = logits[:, 77:83]
        return preds


In [5]:
model = CLIPMultiClassPartitioned(num_classes=NUM_CLASSES).to(device)

# Loss and optimizer (AdamW), MSE loss & CrossEntropy loss
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Checkpoint
ckpt_dir = Path("./checkpoints-clip")
ckpt_dir.mkdir(exist_ok=True)
best_acc = 0.0        # highest val accuracy seen so far

# Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3, verbose=True
)

best_ckpt = ckpt_dir / "best.ckpt"
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optim"])
    scheduler.load_state_dict(ckpt["sched"])
    best_acc = ckpt["val_acc"]
    start_epoch = ckpt["epoch"] + 1
    print(f"✓ Resumed from epoch {start_epoch} with best_acc={best_acc:.2f}%")
else:
    start_epoch = 0

In [6]:
# Training loop, perform test for every 10 epochs, calculate accuracy for multiclass classification (each partition)

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        images = images.to(device)

        optimizer.zero_grad()
        logits = model(images)
        
        preds = model.partition_predictions(logits)

        loss_total = 0.0

        # type: 0–5
        preds_type = preds['type'] # [0-5]
        targets_type = labels['type'].to(device) # [3]
        # print(f"DEBUGGING")
        # ############# DEBUGGING #############
        
        # # print preds_type check type and targets_type check type 
        # print(f"preds_type: {preds_type}, targets_type: {targets_type}")
        # # check if preds_type and targets_type are same shape
        # print(f"preds_type shape: {preds_type.shape}, targets_type shape: {targets_type.shape}")
        # # check if preds_type and targets_type are same dtype
        # print(f"preds_type dtype: {preds_type.dtype}, targets_type dtype: {targets_type.dtype}")
        
        loss_type = criterion(preds_type, targets_type)
        loss_total += loss_type

        # number_of_qubit: 6-35
        preds_qubit = preds['number_of_qubit']
        targets_qubit = labels['number_of_qubit'].to(device)
        loss_qubit = criterion(preds_qubit, targets_qubit)
        loss_total += loss_qubit

        # alpha: 36-51
        preds_alpha = preds['alpha']
        targets_alpha = labels['alpha'].to(device)
        loss_alpha = criterion(preds_alpha, targets_alpha)
        loss_total += loss_alpha

        # number_of_photons: 52-65
        preds_photon = preds['number_of_photons']
        targets_photon = labels['number_of_photons'].to(device)
        loss_photon = criterion(preds_photon, targets_photon)
        loss_total += loss_photon

        # density: 66-76
        preds_density = preds['density']
        targets_density = labels['density'].to(device)
        loss_density = criterion(preds_density, targets_density)
        loss_total += loss_density
            
        # linspace: 77-82
        preds_linspace = preds['linspace']
        targets_linspace = labels['linspace'].to(device)
        loss_linspace = criterion(preds_linspace, targets_linspace)
        loss_total += loss_linspace

        loss_total.backward()
        optimizer.step()
        running_loss += loss_total.item()

    print(f"[Epoch {epoch+1}] Train Loss: {running_loss / len(train_loader):.4f}")

     # ✅ Evaluate every 2 epochs
    if (epoch) % 5 == 0:
        model.eval()

        correct_total = 0
        total_total = 0

        # Partition-wise tracking
        partition_correct = [0] * 6
        partition_total = [0] * 6

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                logits = model(images)
                
                preds = model.partition_predictions(logits)

                # type: 0–5
                preds_type = preds['type'].argmax(1)
                targets_type = labels['type'].to(device)
                correct = (preds_type == targets_type).sum().item()
                partition_correct[0] += correct
                partition_total[0] += BATCH_SIZE
                

                # number_of_qubit: 6-35
                preds_type = preds['number_of_qubit'].argmax(1)
                targets_type = labels['number_of_qubit'].to(device)
                correct = (preds_type == targets_type).sum().item()
                partition_correct[1] += correct
                partition_total[1] += BATCH_SIZE

                
                # alpha: 36-51
                preds_alpha = preds['alpha'].argmax(1)
                targets_alpha = labels['alpha'].to(device)
                correct = (preds_alpha == targets_alpha).sum().item()
                partition_correct[2] += correct
                partition_total[2] += BATCH_SIZE

                
                # number_of_photons: 52-65
                preds_photon = preds['number_of_photons'].argmax(1)
                targets_photon = labels['number_of_photons'].to(device)
                correct = (preds_photon == targets_photon).sum().item()
                partition_correct[3] += correct
                partition_total[3] += BATCH_SIZE
                
                # density: 66-76
                preds_density = preds['density'].argmax(1)
                targets_density = labels['density'].to(device)
                correct = (preds_density == targets_density).sum().item()
                partition_correct[4] += correct
                partition_total[4] += BATCH_SIZE

                # linspace: 77-82
                preds_linspace = preds['linspace'].argmax(1)
                targets_linspace = labels['linspace'].to(device)
                correct = (preds_linspace == targets_linspace).sum().item()
                partition_correct[5] += correct
                partition_total[5] += BATCH_SIZE

        # Compute per-partition accuracy
        print(f"\n🧪 [Epoch {epoch+1}] Partitioned Accuracy:")
        for i in range(6):
            if partition_total[i] > 0:
                acc = 100. * partition_correct[i] / partition_total[i]
                print(f"  Partition {i+1}: {acc:.2f}% ({partition_correct[i]}/{partition_total[i]})")
            else:
                print(f"  Partition {i+1}: No samples")

        # Total accuracy
        correct_total = sum(partition_correct)
        total_total = sum(partition_total)
        total_acc = 100. * correct_total / total_total
        print(f"  ➤ Overall Accuracy: {total_acc:.2f}%")
        
        # ►► step LR scheduler & save checkpoints ◄◄
        scheduler.step(total_acc)                    # adjust LR if plateau
        
        # save rolling checkpoint every evaluation
        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "optim": optimizer.state_dict(),
                "sched": scheduler.state_dict(),
                "val_acc": total_acc,
            },
            ckpt_dir / "last.ckpt",
        )

        # save the best checkpoint
        if total_acc > best_acc:
            best_acc = total_acc
            torch.save(
                {
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "optim": optimizer.state_dict(),
                    "sched": scheduler.state_dict(),
                    "val_acc": best_acc,
                },
                ckpt_dir / "best.ckpt",
            )
            print(f"  ✔ New best model saved  (val_acc = {best_acc:.2f}%)")

Epoch 1/150: 100%|██████████| 883/883 [20:51<00:00,  1.42s/it]


[Epoch 1] Train Loss: 9.7897

🧪 [Epoch 1] Partitioned Accuracy:
  Partition 1: 68.72% (1215/1768)
  Partition 2: 5.49% (97/1768)
  Partition 3: 52.43% (927/1768)
  Partition 4: 67.59% (1195/1768)
  Partition 5: 84.62% (1496/1768)
  Partition 6: 20.31% (359/1768)
  ➤ Overall Accuracy: 49.86%
  ✔ New best model saved  (val_acc = 49.86%)


Epoch 2/150: 100%|██████████| 883/883 [21:17<00:00,  1.45s/it]


[Epoch 2] Train Loss: 8.8216


Epoch 3/150: 100%|██████████| 883/883 [20:55<00:00,  1.42s/it]


...

[Epoch 12] Train Loss: 8.6685


Epoch 13/150: 100%|██████████| 883/883 [20:55<00:00,  1.42s/it]


[Epoch 13] Train Loss: 8.3835


Epoch 14/150:  42%|████▏     | 375/883 [09:00<12:12,  1.44s/it]


KeyboardInterrupt: 